<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

## Step 10.0 — Setup

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = Path("tutorials")
DATA_FOLDER = TUTORIALS_FOLDER / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = TUTORIALS_FOLDER / "yolov8n.pt"

# From .env (or hardcoded fallback for Colab)
PHONE_IP = os.environ.get("PHONE_IP", "192.168.1.42")
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = os.environ.get("SLACK_WEBHOOK", "")
HUGGINGFACE_API_KEY = os.environ.get("HUGGINGFACE_API_KEY", "")

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Phone URL: {PHONE_URL}")

Snapshot folder: tutorials/data/snapshots
Phone URL: http://192.168.1.207:8080/photo.jpg


## Step 10.1 — Query the last 24 hours

`list_sightings_from_db` takes a `since=` timestamp. Compute it from `datetime.now()`.

In [ ]:
from datetime import datetime, timedelta
from bird_watcher.save_sighting import list_sightings_from_db

cutoff = (datetime.now() - timedelta(hours=24)).isoformat(timespec="seconds")
recent = list_sightings_from_db(DB_FILE, since=cutoff, limit=500, verbose=False)
print(f"{len(recent)} sighting(s) in the last 24h")

3 sighting(s) in the last 24h


## Step 10.2 — Group by species, find the top visitor

In [ ]:
from collections import Counter

counts = Counter(s["species"] for s in recent)
print(f"{len(counts)} unique species")
for species, count in counts.most_common(5):
    print(f"  {species}: {count}")

1 unique species
  Northern Cardinal: 3


## Step 10.3 — Wrap as `build_daily_summary`

In [0]:
#| echo: false
#| output: asis
show_doc(build_daily_summary)

---

### build_daily_summary

```python
def build_daily_summary(
    db_file:Path, snapshot_folder:Path, window_hours:int=24
)->dict:
```

*Build a Slack message summarizing the last `window_hours` of sightings.*

Args:
    db_file: where sightings are stored.
    snapshot_folder: where snapshot images live (for referencing).
    window_hours: how far back to look. Default 24 (one day).

Returns:
    A Slack Block Kit payload.

## Step 10.4 — Add `send_daily_summary`

In [0]:
#| echo: false
#| output: asis
show_doc(send_daily_summary)

---

### send_daily_summary

```python
def send_daily_summary(
    db_file:Path, snapshot_folder:Path, window_hours:int=24, webhook_url:str | None=None, verbose:bool=True
)->bool:
```

*Build and send (or dry-run) today's daily summary.*

Args:
    db_file: where sightings are stored.
    snapshot_folder: where snapshot images live.
    window_hours: how far back to look.
    webhook_url: optional Slack webhook URL. Uses env var otherwise.
    verbose: if True, print a status line.

Returns:
    True if sent (or dry-run previewed), False on error.

## Step 10.5 — And the scheduler `schedule_daily_summary`

In [0]:
#| echo: false
#| output: asis
show_doc(schedule_daily_summary)

---

### schedule_daily_summary

```python
def schedule_daily_summary(
    db_file:Path, snapshot_folder:Path, run_at_hour:int=21, verbose:bool=True
)->None:
```

*Run a blocking scheduler that posts the daily summary once a day.*

For learning purposes only — production code would use a real cron job.

Args:
    db_file: where sightings are stored.
    snapshot_folder: where snapshot images live.
    run_at_hour: hour of day to send (24h format). Default 21 (9pm).
    verbose: if True, print the next scheduled run.

## Acceptance criterion

In [ ]:
from bird_watcher.daily_summary import build_daily_summary, send_daily_summary

payload = build_daily_summary(DB_FILE, SNAPSHOT_FOLDER)
assert "blocks" in payload
assert any(b.get("type") == "header" for b in payload["blocks"])

sent = send_daily_summary(DB_FILE, SNAPSHOT_FOLDER)
assert sent, "send_daily_summary should return True (dry-run if no webhook)"
print("✅ Daily summary built + sent (or dry-run)")

[dry-run] No SLACK_WEBHOOK set. Would have sent:
{
  "blocks": [
    {
      "type": "header",
      "text": {
        "type": "plain_text",
        "text": ":bird: Daily summary \u2014 3 sighting(s), 1 species, top: *Northern Cardinal*"
      }
    },
    {
      "type": "section",
      "fields": [
        {
          "type": "mrkdwn",
          "text": "*Window*\nlast 24h"
        },
        {
          "type": "mrkdwn",
          "text": "*Total sightings*\n3"
        },
        {
          "type": "mrkdwn",
          "text": "*Unique species*\n1"
        }
      ]
    },
    {
      "type": "section",
      "text": {
        "type": "mrkdwn",
        "text": "*Top species*\n\u2022 Northern Cardinal \u2014 3"
      }
    }
  ]
}
✅ Daily summary built + sent (or dry-run)


## 🎉 You finished the bird watcher tutorial!

Ten notebooks. Seven modules. One bird watcher.

- pull photos from a phone camera
- find birds in those photos
- name the species
- log everything to a database
- post alerts to Slack
- serve a web gallery
- send a daily digest

**What's next?** Experiment. Add features. Break stuff. That's how you learn.